In [ ]:
# IMPORT LIBRARIES
import torch
import matplotlib.pyplot as plt
import pandas as pd

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# CHECK AVAILABLE DEVICE
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
device

In [ ]:
!unzip -q "/content/Directions01.zip" -d "/content/"

In [ ]:
# IMAGE PREPROCESSING FOR PRETRAINED RESNET-18

transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# LOAD TRAIN AND TEST DATASETS
train_data = datasets.ImageFolder(
    root="/content/Directions01/train",
    transform=transform
)

test_data = datasets.ImageFolder(
    root="/content/Directions01/test",
    transform=transform
)

print("Training images:", len(train_data))
print("Test images:", len(test_data))
print("Classes:", train_data.classes)
print("Class mapping:", train_data.class_to_idx)

In [ ]:
# VISUALIZE TRAINING IMAGES

figure = plt.figure(figsize=(10, 8))
cols, rows = 4, 4

for i in range(1, cols * rows + 1):

    sample_idx = torch.randint(len(train_data), size=(1,)).item()

    img, label = train_data[sample_idx]

    # Use first channel for grayscale visualization
    img_display = img[0]

    figure.add_subplot(rows, cols, i)
    plt.title(train_data.classes[label])
    plt.axis("off")
    plt.imshow(img_display, cmap="gray")

plt.show()

In [ ]:
# Preparing data for training with DataLoaders
from torch.utils.data import DataLoader
loaders = {
    'train': DataLoader(
        train_data,
        batch_size=32,
        shuffle=True,
        num_workers=2
    ),

    'test': DataLoader(
        test_data,
        batch_size=32,
        shuffle=False,
        num_workers=2
    )
}

loaders

In [ ]:
# CREATING THE CNN MODEL (PRETRAINED RESNET-18)

import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

# Load ResNet-18 with pretrained ImageNet weights
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)

# Replace final classification layer
# ImageNet = 1000 classes
# X-ray orientation dataset = 4 classes
model.fc = nn.Linear(
    model.fc.in_features,
    4
)

# Move model to GPU if available
model = model.to(device)

print(model)

In [ ]:
# Loss Function and Optimizer
import torch.optim as optim

loss_func = nn.CrossEntropyLoss()

learning_rate = 0.0001

optimizer = optim.Adam(
    model.parameters(),
    lr=learning_rate
)
optimizer

In [ ]:
# Training Function

def train(epoch):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (images, labels) in enumerate(loaders['train']):

        images = images.to(device)
        labels = labels.to(device)

        # Reset gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = loss_func(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update model parameters
        optimizer.step()

        running_loss += loss.item()

        # Calculate training accuracy
        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = running_loss / len(loaders['train'])
    train_accuracy = 100 * correct / total

    print(
        f'Epoch {epoch}: '
        f'Loss = {train_loss:.4f}, '
        f'Training Accuracy = {train_accuracy:.2f}%'
    )

    return train_loss, train_accuracy

In [ ]:
# Testing Function

def test():
    model.eval()

    correct = 0
    total = 0
    test_loss = 0.0

    with torch.no_grad():

        for images, labels in loaders['test']:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = loss_func(outputs, labels)

            test_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    test_loss /= len(loaders['test'])
    test_accuracy = 100 * correct / total

    print(
        f'Test Loss = {test_loss:.4f}, '
        f'Test Accuracy = {test_accuracy:.2f}%'
    )

    return test_loss, test_accuracy

In [ ]:
# ------------------------------
# Train ResNet-18 and save results
# ------------------------------

import pandas as pd

num_epochs = 10

results = []

for epoch in range(1, num_epochs + 1):

    train_loss, train_accuracy = train(epoch)
    test_loss, test_accuracy = test()

    results.append({
        'Epoch': epoch,
        'Learning Rate': learning_rate,
        'Train Loss': train_loss,
        'Train Accuracy': train_accuracy,
        'Test Loss': test_loss,
        'Test Accuracy': test_accuracy
    })

In [ ]:
# ------------------------------
# Show 10 predictions
# ------------------------------

model.eval()

images, labels = next(iter(loaders['test']))

images = images.to(device)
labels = labels.to(device)

with torch.no_grad():
    outputs = model(images)
    _, predictions = torch.max(outputs, 1)

# Convert class numbers to class names
predicted_classes = [test_data.classes[p] for p in predictions[:10].cpu()]
actual_classes = [test_data.classes[l] for l in labels[:10].cpu()]

print("Predicted:", predicted_classes)
print("Actual:   ", actual_classes)

In [ ]:
# ------------------------------
# Save the trained model and results as a CSV
# ------------------------------

results_df = pd.DataFrame(results)

results_df.to_csv(
    'resnet18_grayscale_epochs10_results.csv',
    index=False
)

results_df

torch.save({
    'model': 'ResNet-18',
    'image_type': 'Grayscale 8-bit',
    'epochs': num_epochs,
    'learning_rate': learning_rate,

    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),

    'results': results
},
'resnet18_grayscale_epochs10.pth')

In [ ]:
# UNZIP RGB AND INDEXED COLOR DATASETS

!unzip -q "/content/Directions01_RGB.zip" -d "/content/"
!unzip -q "/content/Directions01_Index.zip" -d "/content/"

import os

print(os.listdir("/content/Directions01_RGB"))
print(os.listdir("/content/Directions01_Index"))

In [ ]:
# ------------------------------
# RGB 24-BIT DATASET
# ------------------------------

rgb_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

rgb_train_data = datasets.ImageFolder(
    root="/content/Directions01_RGB/train",
    transform=rgb_transform
)

rgb_test_data = datasets.ImageFolder(
    root="/content/Directions01_RGB/test",
    transform=rgb_transform
)

print("Training images:", len(rgb_train_data))
print("Test images:", len(rgb_test_data))
print("Classes:", rgb_train_data.classes)
print("Class mapping:", rgb_train_data.class_to_idx)

In [ ]:
rgb_loaders = {
    'train': DataLoader(
        rgb_train_data,
        batch_size=32,
        shuffle=True,
        num_workers=2
    ),

    'test': DataLoader(
        rgb_test_data,
        batch_size=32,
        shuffle=False,
        num_workers=2
    )
}

In [ ]:
# ------------------------------
# RGB RESNET-18
# ------------------------------

from torchvision.models import resnet18, ResNet18_Weights
import torch.nn as nn
import torch.optim as optim

weights = ResNet18_Weights.DEFAULT

rgb_model = resnet18(weights=weights)

rgb_model.fc = nn.Linear(
    rgb_model.fc.in_features,
    4
)

rgb_model = rgb_model.to(device)

rgb_loss_func = nn.CrossEntropyLoss()

rgb_optimizer = optim.Adam(
    rgb_model.parameters(),
    lr=learning_rate
)

In [ ]:
# ------------------------------
# TRAINING FUNCTION FOR NEW EXPERIMENTS
# ------------------------------

def train_model(model, loader, optimizer, loss_func):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = loss_func(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_accuracy = 100 * correct / total

    return epoch_loss, epoch_accuracy


def test_model(model, loader, loss_func):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = loss_func(outputs, labels)

            running_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_accuracy = 100 * correct / total

    return epoch_loss, epoch_accuracy

In [ ]:
# ------------------------------
# TRAIN RGB MODEL
# ------------------------------

num_epochs = 10

rgb_results = []

for epoch in range(1, num_epochs + 1):

    train_loss, train_accuracy = train_model(
        rgb_model,
        rgb_loaders['train'],
        rgb_optimizer,
        rgb_loss_func
    )

    test_loss, test_accuracy = test_model(
        rgb_model,
        rgb_loaders['test'],
        rgb_loss_func
    )

    print(
        f"Epoch {epoch}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Accuracy: {train_accuracy:.2f}% | "
        f"Test Loss: {test_loss:.4f} | "
        f"Test Accuracy: {test_accuracy:.2f}%"
    )

    rgb_results.append({
        'Epoch': epoch,
        'Learning Rate': learning_rate,
        'Train Loss': train_loss,
        'Train Accuracy': train_accuracy,
        'Test Loss': test_loss,
        'Test Accuracy': test_accuracy
    })

In [ ]:
# ------------------------------
# SHOW 10 RGB PREDICTIONS
# ------------------------------

rgb_model.eval()

images, labels = next(iter(rgb_loaders['test']))

images = images.to(device)
labels = labels.to(device)

with torch.no_grad():

    outputs = rgb_model(images)

    _, predictions = torch.max(outputs, 1)

predicted_classes = [
    rgb_test_data.classes[p]
    for p in predictions[:10].cpu()
]

actual_classes = [
    rgb_test_data.classes[l]
    for l in labels[:10].cpu()
]

print("Predicted:", predicted_classes)
print("Actual:   ", actual_classes)

In [ ]:
# ------------------------------
# SAVE RGB RESULTS AND MODEL
# ------------------------------

rgb_results_df = pd.DataFrame(rgb_results)

rgb_results_df.to_csv(
    'resnet18_rgb_epochs10_results.csv',
    index=False
)

rgb_results_df

torch.save({
    'model': 'ResNet-18',
    'image_type': 'RGB 24-bit',
    'epochs': num_epochs,
    'learning_rate': learning_rate,

    'model_state_dict': rgb_model.state_dict(),
    'optimizer_state_dict': rgb_optimizer.state_dict(),

    'results': rgb_results
},
'resnet18_rgb_epochs10.pth')

In [ ]:
# ------------------------------
# INDEXED COLOR 8-BIT DATASET
# ------------------------------

index_transform = transforms.Compose([
    transforms.Lambda(lambda img: img.convert("RGB")),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

index_train_data = datasets.ImageFolder(
    root="/content/Directions01_Index/train",
    transform=index_transform
)

index_test_data = datasets.ImageFolder(
    root="/content/Directions01_Index/test",
    transform=index_transform
)

print("Training images:", len(index_train_data))
print("Test images:", len(index_test_data))
print("Classes:", index_train_data.classes)
print("Class mapping:", index_train_data.class_to_idx)

In [ ]:
index_loaders = {
    'train': DataLoader(
        index_train_data,
        batch_size=32,
        shuffle=True,
        num_workers=2
    ),

    'test': DataLoader(
        index_test_data,
        batch_size=32,
        shuffle=False,
        num_workers=2
    )
}

In [ ]:
# ------------------------------
# INDEXED COLOR RESNET-18
# ------------------------------

index_model = resnet18(weights=weights)

index_model.fc = nn.Linear(
    index_model.fc.in_features,
    4
)

index_model = index_model.to(device)

index_loss_func = nn.CrossEntropyLoss()

index_optimizer = optim.Adam(
    index_model.parameters(),
    lr=learning_rate
)

In [ ]:
# ------------------------------
# INDEXED COLOR RESNET-18
# ------------------------------

index_model = resnet18(weights=weights)

index_model.fc = nn.Linear(
    index_model.fc.in_features,
    4
)

index_model = index_model.to(device)

index_loss_func = nn.CrossEntropyLoss()

index_optimizer = optim.Adam(
    index_model.parameters(),
    lr=learning_rate
)

In [ ]:
# ------------------------------
# TRAIN INDEXED COLOR MODEL
# ------------------------------

index_results = []

for epoch in range(1, num_epochs + 1):

    train_loss, train_accuracy = train_model(
        index_model,
        index_loaders['train'],
        index_optimizer,
        index_loss_func
    )

    test_loss, test_accuracy = test_model(
        index_model,
        index_loaders['test'],
        index_loss_func
    )

    print(
        f"Epoch {epoch}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Accuracy: {train_accuracy:.2f}% | "
        f"Test Loss: {test_loss:.4f} | "
        f"Test Accuracy: {test_accuracy:.2f}%"
    )

    index_results.append({
        'Epoch': epoch,
        'Learning Rate': learning_rate,
        'Train Loss': train_loss,
        'Train Accuracy': train_accuracy,
        'Test Loss': test_loss,
        'Test Accuracy': test_accuracy
    })

In [ ]:
# ------------------------------
# SHOW 10 INDEXED COLOR PREDICTIONS
# ------------------------------

index_model.eval()

images, labels = next(iter(index_loaders['test']))

images = images.to(device)
labels = labels.to(device)

with torch.no_grad():

    outputs = index_model(images)

    _, predictions = torch.max(outputs, 1)

predicted_classes = [
    index_test_data.classes[p]
    for p in predictions[:10].cpu()
]

actual_classes = [
    index_test_data.classes[l]
    for l in labels[:10].cpu()
]

print("Predicted:", predicted_classes)
print("Actual:   ", actual_classes)

In [ ]:
# ------------------------------
# SAVE INDEXED COLOR RESULTS AND MODEL
# ------------------------------

index_results_df = pd.DataFrame(index_results)

index_results_df.to_csv(
    'resnet18_index_epochs10_results.csv',
    index=False
)

index_results_df

torch.save({
    'model': 'ResNet-18',
    'image_type': 'Indexed Color 8-bit',
    'epochs': num_epochs,
    'learning_rate': learning_rate,

    'model_state_dict': index_model.state_dict(),
    'optimizer_state_dict': index_optimizer.state_dict(),

    'results': index_results
},
'resnet18_index_epochs10.pth')

In [ ]:
# ------------------------------
# COMPARISON OF ACCURACY
# ------------------------------

plt.figure(figsize=(9, 6))

plt.plot(
    results_df['Epoch'],
    results_df['Train Accuracy'],
    marker='o',
    label='Grayscale - Train'
)

plt.plot(
    results_df['Epoch'],
    results_df['Test Accuracy'],
    marker='o',
    linestyle='--',
    label='Grayscale - Test'
)

plt.plot(
    rgb_results_df['Epoch'],
    rgb_results_df['Train Accuracy'],
    marker='o',
    label='RGB - Train'
)

plt.plot(
    rgb_results_df['Epoch'],
    rgb_results_df['Test Accuracy'],
    marker='o',
    linestyle='--',
    label='RGB - Test'
)

plt.plot(
    index_results_df['Epoch'],
    index_results_df['Train Accuracy'],
    marker='o',
    label='Indexed Color - Train'
)

plt.plot(
    index_results_df['Epoch'],
    index_results_df['Test Accuracy'],
    marker='o',
    linestyle='--',
    label='Indexed Color - Test'
)

plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('ResNet-18 Accuracy Comparison')
plt.legend()
plt.grid(True)

plt.show()


# ------------------------------
# COMPARISON OF LOSS
# ------------------------------

plt.figure(figsize=(9, 6))

plt.plot(
    results_df['Epoch'],
    results_df['Train Loss'],
    marker='o',
    label='Grayscale - Train'
)

plt.plot(
    results_df['Epoch'],
    results_df['Test Loss'],
    marker='o',
    linestyle='--',
    label='Grayscale - Test'
)

plt.plot(
    rgb_results_df['Epoch'],
    rgb_results_df['Train Loss'],
    marker='o',
    label='RGB - Train'
)

plt.plot(
    rgb_results_df['Epoch'],
    rgb_results_df['Test Loss'],
    marker='o',
    linestyle='--',
    label='RGB - Test'
)

plt.plot(
    index_results_df['Epoch'],
    index_results_df['Train Loss'],
    marker='o',
    label='Indexed Color - Train'
)

plt.plot(
    index_results_df['Epoch'],
    index_results_df['Test Loss'],
    marker='o',
    linestyle='--',
    label='Indexed Color - Test'
)

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('ResNet-18 Loss Comparison')
plt.legend()
plt.grid(True)

plt.show()


